# PIV Analysis of the Tracked Sequences

This notebook consumes the cropped images produced by `opencv_tracker_v3.py` and:

1. Reads the saved Otsu threshold to build a mask of the tracked object.
2. Expands the mask to remove the near-field region before running OpenPIV on each frame pair.
3. Stores the resulting vector fields (`.npz`) and a CSV summarizing the mean/std speed.
4. Provides an interactive slider to overlay velocity vectors on the source image for verification.


In [ ]:
import csv
import json
import re
from importlib import import_module
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
if not hasattr(np, 'int'):
    np.int = int
import pandas as pd
import yaml
from IPython.display import display

plt.rcParams['figure.figsize'] = (6, 6)
CONFIG_PATH = Path('../configs/opencv_tracker_v3.yaml')

def _import_module(name, fallback=None):
    try:
        return import_module(name)
    except ModuleNotFoundError:
        if fallback:
            return import_module(fallback)
        raise

try:
    openpiv = import_module('openpiv')
    filters = _import_module('openpiv.filters')
    process_module = _import_module('openpiv.process', 'openpiv.pyprocess')
    scaling = _import_module('openpiv.scaling')
    validation = _import_module('openpiv.validation')
    tools = _import_module('openpiv.tools')
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "OpenPIV is required for the PIV cells. Install it with `pip install openpiv==0.23.4 using the same Python kernel you run this notebook with, then restart the kernel."
    ) from exc

openpiv.filters = filters
openpiv.process = process_module
openpiv.scaling = scaling
openpiv.validation = validation
openpiv.tools = tools






In [ ]:
def resolve_path(path_like, base_dir: Path) -> Path:
    candidate = Path(path_like)
    return candidate if candidate.is_absolute() else (base_dir / candidate).resolve()

def natural_sort_key(path: Path):
    parts = re.split(r'(\d+)', path.name)
    return [int(part) if part.isdigit() else part.lower() for part in parts]

def collect_images(folder: Path, allowed_exts=None):
    allowed_exts = [ext.lower() for ext in (allowed_exts or [])]
    candidates = [
        p
        for p in folder.iterdir()
        if p.is_file()
        and not p.name.startswith('.')
    ]
    if allowed_exts:
        candidates = [p for p in candidates if p.suffix.lower() in allowed_exts]
    return sorted(candidates, key=natural_sort_key)

def load_experiment_metadata(experiment_dir: Path) -> dict:
    for suffix in ('metadata.json', 'metadata.csv'):
        path = experiment_dir / suffix
        if path.exists():
            if path.suffix.lower() == '.json':
                return json.loads(path.read_text(encoding='utf-8'))
            with path.open('r', encoding='utf-8') as handle:
                reader = csv.DictReader(handle)
                for row in reader:
                    return row
    return {}




In [ ]:
# Load config files and paths

config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
config_dir = CONFIG_PATH.parent
input_cfg = config['input']
base_dir = resolve_path(input_cfg['base_dir'], config_dir)
raw_root_candidate = input_cfg.get('raw_root')
raw_root = resolve_path(raw_root_candidate or base_dir.parent, config_dir)
processed_root_candidate = input_cfg.get('processed_root')
fallback_processed = raw_root.parent / 'processed'
processed_root = resolve_path(processed_root_candidate or fallback_processed, config_dir)
try:
    relative = base_dir.relative_to(raw_root)
except ValueError:
    relative = Path(base_dir.name)
processed_experiment_dir = processed_root / relative
crops_subdir = config['output'].get('crops_subdir', 'cropped') or 'cropped'
crops_root = processed_experiment_dir / crops_subdir

piv_root = processed_experiment_dir / 'piv_data'
piv_root.mkdir(parents=True, exist_ok=True)
summary_path = processed_experiment_dir / config['output'].get('summary_json', 'frame_rate_summary.json')
summary_data = json.loads(summary_path.read_text(encoding='utf-8'))
folder_summaries = summary_data.get('folder_summaries', [])
metadata_record = load_experiment_metadata(processed_experiment_dir)
tracking_csv = processed_experiment_dir / config['output'].get('metadata_csv', 'tracking_metadata.csv')
tracking_df = pd.read_csv(tracking_csv) if tracking_csv.exists() else pd.DataFrame()
print(f'Processed data directory: {processed_experiment_dir}')
print(f'Loaded {len(folder_summaries)} folder summaries and {len(tracking_df)} tracking rows.')


In [ ]:
def build_far_field_mask(image: np.ndarray, threshold: float, iterations: int = 5):
    _, mask = cv2.threshold(image, int(threshold), 255, cv2.THRESH_BINARY)
    if not np.any(mask):
        _, mask = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    expanded = cv2.dilate(mask, kernel, iterations=iterations)
    far_field = cv2.bitwise_not(expanded)
    return far_field, expanded

def histogram_stretch(image: np.ndarray, lower_pct: float = 0.1, upper_pct: float = 99.9) -> np.ndarray:
    arr = np.asarray(image, dtype=np.float32)
    if arr.ndim != 2:
        arr = cv2.cvtColor(arr, cv2.COLOR_BGR2GRAY)
    low = np.percentile(arr, lower_pct)
    high = np.percentile(arr, upper_pct)
    if high <= low:
        return arr.astype(np.uint8)
    stretched = (arr - low) * (255.0 / (high - low))
    stretched = np.clip(stretched, 0, 255).astype(np.uint8)
    return stretched

def even_window_size(value: int, minimum: int = 32) -> int:
    sanitized = max(minimum, value)
    if sanitized % 2 != 0:
        sanitized -= 1
    return max(sanitized, minimum)

def mask_vectors_by_image_mask(x, y, u, v, mask):
    mask_arr = np.asarray(mask)
    if mask_arr.ndim != 2:
        return u, v
    object_mask = mask_arr == 0
    h, w = object_mask.shape
    xi = np.clip(np.round(x).astype(int), 0, w - 1)
    yi = np.clip(np.round(y).astype(int), 0, h - 1)
    inside_object = object_mask[yi, xi]
    u_out = np.asarray(u, dtype=float).copy()
    v_out = np.asarray(v, dtype=float).copy()
    u_out[inside_object] = np.nan
    v_out[inside_object] = np.nan
    return u_out, v_out





In [ ]:
# PIV paramters
piv_window_scale_factor = 8 # window_size = even_window_size(min_dim // piv_window_scale_factor, minimum=32)
search_area_scale_factor = 2 # search_area_size = window_size * search_area_scale_factor
validation_method = 'localmean'



In [ ]:
piv_artifacts = {}
piv_summary_rows = []
save_ext = config['output'].get('save_extension', '.png').lower()
allowed_exts = [save_ext]

for folder_summary in folder_summaries:
    folder_name = folder_summary.get('subfolder')
    threshold_value = folder_summary.get('threshold_value')
    frame_rate = folder_summary.get('frame_rate_hz')
    if not folder_name or threshold_value is None or not frame_rate:
        print(f"Skipping '{folder_name}' because of missing threshold/frame rate.")
        continue
    cropped_folder = crops_root / f"{folder_name}_cropped"
    if not cropped_folder.exists():
        print(f"Missing cropped folder: {cropped_folder}")
        continue
    image_paths = collect_images(cropped_folder, allowed_exts)
    if len(image_paths) < 2:
        print(f"Not enough frames in {cropped_folder} to compute PIV.")
        continue
    grayscale_images = [cv2.imread(str(path), cv2.IMREAD_GRAYSCALE) for path in image_paths]
    far_field_mask, expanded_mask = build_far_field_mask(
        grayscale_images[0], threshold_value, iterations=5
    )
    far_field_mask = far_field_mask.astype('uint8', copy=False)
    dt = 1.0 / frame_rate
    min_dim = min(image.shape[0] for image in grayscale_images)

    window_size = int(even_window_size(min_dim // piv_window_scale_factor, minimum=32))
    overlap = max(4, window_size // 2)
    search_area_size = int(max(window_size * search_area_scale_factor, window_size + 8))
    
    pixel_per_um = float(metadata_record.get('pixelperum', 1.36)) or 1.36
    piv_subdir = piv_root / f"{folder_name}_piv_data"
    piv_subdir.mkdir(parents=True, exist_ok=True)

    magnitude_frames = []
    npz_paths = []
    pair_stats = []

    for idx in range(len(grayscale_images) - 1):
        frame_a_path = image_paths[idx]
        frame_b_path = image_paths[idx + 1]
        print(f'Frame A: {frame_a_path.name} / Frame B: {frame_b_path.name}')
        frame_a = histogram_stretch(grayscale_images[idx]).astype(np.int32)
        frame_b = histogram_stretch(grayscale_images[idx + 1]).astype(np.int32)
        u, v, sig2noise = openpiv.process.extended_search_area_piv(
            frame_a.astype(np.int32),
            frame_b.astype(np.int32),
            window_size=window_size,
            overlap=overlap,
            dt=dt,
            search_area_size=search_area_size,
            sig2noise_method='peak2peak',
        )
        x, y = openpiv.process.get_coordinates(image_size=frame_a.shape,
                                                search_area_size=search_area_size,
                                                overlap=overlap)
        u, v, mask = openpiv.validation.sig2noise_val(u, v, sig2noise, threshold=1.1)
        # u, v = openpiv.filters.replace_outliers(
        #     u, v, method='localmean', max_iter=10, kernel_size=2
        # )
        u, v = mask_vectors_by_image_mask(x, y, u, v, far_field_mask)
        x, y, u, v = openpiv.scaling.uniform(
            x, y, u, v, scaling_factor=pixel_per_um
        )
        x, y, u, v = openpiv.tools.transform_coordinates(x, y, u, v)
        mag = np.sqrt(u * u + v * v)
        print(mag)
        pair_valid = ~np.isnan(mag)
        pair_mean = float(np.nan) if not pair_valid.any() else float(np.mean(mag[pair_valid]))
        pair_std = float(np.nan) if not pair_valid.any() else float(np.std(mag[pair_valid]))
        pair_stats.append({
            'pair_index': idx,
            'frame_a': frame_a_path.name,
            'frame_b': frame_b_path.name,
            'mean_velocity': pair_mean,
            'std_velocity': pair_std,
        })
        magnitude_frames.append(mag)
        pair_path = piv_subdir / f"pair_{idx:04d}.npz"
        np.savez_compressed(
            pair_path,
            x=x,
            y=y,
            u=u,
            v=v,
            mask=mask,
            threshold=threshold_value,
            dt=dt,
            window_size=window_size,
        )
        npz_paths.append(pair_path)

    if not magnitude_frames:
        print(f"No usable PIV pairs found for {folder_name}.")
        continue

    stats_path = None
    if pair_stats:
        stats_df = pd.DataFrame(pair_stats)
        stats_path = piv_subdir / 'pair_stats.csv'
        stats_df.to_csv(stats_path, index=False)

    stacked = np.concatenate([frame.flatten() for frame in magnitude_frames])
    valid = ~np.isnan(stacked)
    mean_velocity = float(np.nan) if not valid.any() else float(np.mean(stacked[valid]))
    std_velocity = float(np.nan) if not valid.any() else float(np.std(stacked[valid]))

    piv_artifacts[folder_name] = {
        'images': grayscale_images,
        'npz_paths': npz_paths,
    }

    piv_summary_rows.append({
        'experiment_folder': processed_experiment_dir.name,
        'experiment_path': str(processed_experiment_dir),
        'subfolder': folder_name,
        'particle_type': metadata_record.get('particle_type', 'unknown'),
        'pixelperum': pixel_per_um,
        'object_size_um': folder_summary.get('hydrodynamic_diameter_um'),
        'frame_rate_hz': frame_rate,
        'threshold_value': threshold_value,
        'mean_velocity': mean_velocity,
        'std_velocity': std_velocity,
        'num_pairs': len(npz_paths),
        'window_size': window_size,
        'npz_folder': str(piv_subdir),
        'pair_stats_csv': str(stats_path) if stats_path else None,
    })
    print(f"Stored {len(npz_paths)} PIV pairs for subfolder '{folder_name}'.")




In [ ]:
piv_summary_df = pd.DataFrame(piv_summary_rows)
if not piv_summary_df.empty:
    summary_csv = piv_root / 'piv_velocity_summary.csv'
    piv_summary_df.to_csv(summary_csv, index=False)
    display(piv_summary_df)
    print(f'Saved PIV velocity summary to {summary_csv}')
else:
    print('No PIV summary could be generated yet.')



In [ ]:
if piv_artifacts:
    subfolder_selector = widgets.Dropdown(
        options=list(piv_artifacts.keys()),
        description='Subfolder',
    )
    pair_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=max(len(piv_artifacts[subfolder_selector.value]['npz_paths']) - 1, 0),
        step=1,
        description='Pair index',
    )

    def update_slider_range(change):
        key = change['new']
        available = piv_artifacts.get(key, {'npz_paths': []})['npz_paths']
        max_index = max(len(available) - 1, 0)
        pair_slider.max = max_index
        pair_slider.value = 0

    subfolder_selector.observe(update_slider_range, names='value')

    def render_velocity_field(subfolder, pair_index):
        data = piv_artifacts[subfolder]
        if not data['npz_paths']:
            print('No PIV files saved for this subfolder yet.')
            return
        pair_index = min(pair_index, len(data['npz_paths']) - 1)
        image = data['images'][pair_index]
        with np.load(str(data['npz_paths'][pair_index])) as pivot:
            x = pivot['x']
            y = pivot['y']
            u = pivot['u']
            v = pivot['v']
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(image, cmap='gray')
        ax.quiver(x, y, u, v, color='red')
        ax.set_title(f"{subfolder} | pair {pair_index}")
        ax.set_axis_off()
        plt.show()

    out = widgets.interactive_output(
        render_velocity_field,
        {'subfolder': subfolder_selector, 'pair_index': pair_slider},
    )
    display(widgets.VBox([subfolder_selector, pair_slider, out]))
else:
    print('PIV data is not available yet. Run the processing cell first.')



## PIV stats and diagnostics

In [ ]:
stats_df.head()

In [ ]:
# Plot the mean velocity and std velocity vs pair_index as an error bar plot using matplotlib, with mean velocity in blue and std velocity in orange
plt.figure(figsize=(10, 4))
plt.errorbar(stats_df['pair_index'], stats_df['mean_velocity'], yerr=stats_df['std_velocity'], fmt='o', ecolor='orange', color='blue', capsize=5)
plt.title('Mean Velocity with Standard Deviation Error Bars')
plt.xlabel('Pair Index')
plt.ylabel('Velocity (um/s)')
plt.grid()
plt.show()

## PIV sandbox

In [ ]:
# PIV sandbox

frame_a_path = image_paths[0]
frame_b_path = image_paths[1]

print(f"Frame A: {frame_a_path}")
print(f"Frame B: {frame_b_path}")

frame_a = cv2.imread(str(frame_a_path), cv2.IMREAD_GRAYSCALE).astype(np.int32)
frame_b = cv2.imread(str(frame_b_path), cv2.IMREAD_GRAYSCALE).astype(np.int32)

# show the two frames side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(frame_a, cmap='gray')
axes[0].set_title('Frame A')
axes[0].set_axis_off()
axes[1].imshow(frame_b, cmap='gray')
axes[1].set_title('Frame B')
axes[1].set_axis_off()
plt.show()  

In [ ]:
window_size = even_window_size(min(frame_a.shape) // 2, minimum=32)
overlap = max(4, window_size // 2)
search_area_size = max(window_size * 2, window_size + 8)
dt = 1.0 / frame_rate

print(f"Using window size: {window_size}, overlap: {overlap}, search area size: {search_area_size}, dt: {dt}")

In [ ]:
# Run PIV on the two frames
u, v, sig2noise = openpiv.process.extended_search_area_piv(
    frame_a.astype(np.int32),
    frame_b.astype(np.int32),
    window_size=window_size,
    overlap=overlap,
    dt=dt,
    search_area_size=search_area_size,
    sig2noise_method='peak2peak',
)
x, y = openpiv.process.get_coordinates(image_size=frame_a.shape,
    search_area_size=search_area_size,
    overlap=overlap)
# u, v, mask = openpiv.validation.sig2noise_val(u, v, sig2noise, threshold=1.1)
# u, v = openpiv.filters.replace_outliers(
#     u, v, method='localmean', max_iter=10, kernel_size=2
# )
x, y, u, v = openpiv.scaling.uniform(
    x, y, u, v, scaling_factor=pixel_per_um
)   

x, y, u, v = openpiv.tools.transform_coordinates(x, y, u, v)

In [ ]:
print(v)

In [ ]:
print(np.shape(u))
print(np.shape(x))


In [ ]:
# SHow the velocity field
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(frame_a, cmap='gray')
ax.quiver(x, y, u, v, color='red')
ax.set_title('PIV Velocity Field')
ax.set_axis_off()
plt.show()

In [ ]:
v